# 3. Inference Pipeline

**Purpose**: Apply trained model at scale to 11M+ records via chunked processing.

## Sections
1. Load config and saved model
2. Define chunking strategy
3. Process chunks with progress tracking
4. Cluster predictions
5. Export results


---
## 1. Setup and Load Model


In [ ]:
# AWS credentials should be set in your terminal before running:
# export AWS_ACCESS_KEY_ID='your_key'
# export AWS_SECRET_ACCESS_KEY='your_secret'

import os
if not os.getenv('AWS_ACCESS_KEY_ID') or not os.getenv('AWS_SECRET_ACCESS_KEY'):
    print("WARNING: AWS credentials not set. Run in terminal:")
    print("  export AWS_ACCESS_KEY_ID='your_key'")
    print("  export AWS_SECRET_ACCESS_KEY='your_secret'")

In [ ]:
import sys
sys.path.insert(0, '/Users/robertlalani/Desktop/entity_resolution_12_18_25/org_claude_test')

import pandas as pd
import numpy as np
import json
from pathlib import Path

# Splink imports
from splink import Linker, DuckDBAPI

# Local imports
from config import config
from utils import (
    DatabaseManager, 
    log_step, 
    Timer,
    ProgressTracker,
    save_checkpoint,
    load_checkpoint,
    load_json
)
from data_prep import (
    load_mismatched,
    load_dim_org,
    create_unified_schema,
    filter_bad_records,
    create_blocking_keys,
    add_distinctive_tokens,  # Changed from add_idf_based_features
    add_token_set_features,
    compute_token_statistics,
    get_corpus_stopwords
)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)

print("Imports loaded successfully")


In [ ]:
# Initialize database
db = DatabaseManager()

# Load saved model
model_path = config.paths.MODEL_FILE
if not Path(model_path).exists():
    raise FileNotFoundError(f"Model not found at {model_path}. Run 2_training.ipynb first.")

model_json = load_json(model_path, "Trained model")

print(f"\nMODEL LOADED")
print("=" * 50)
print(f"Path: {model_path}")
if 'training_metadata' in model_json:
    meta = model_json['training_metadata']
    print(f"Trained on: {meta.get('timestamp', 'N/A')}")
    print(f"Training records: {meta.get('dim_org_records', 0):,} dim_org + {meta.get('grid_records', 0):,} GRID")


In [ ]:
# Load reference data (dim_org) for linking
# Using cached data if available
cached_dim_org = load_checkpoint(config.paths.DATA_DIR + "/dim_org_training.parquet", "dim_org cache")

if cached_dim_org is not None:
    dim_org_df = cached_dim_org
    print(f"Loaded cached dim_org: {len(dim_org_df):,} records")
else:
    dim_org_raw = load_dim_org(db, sample_size=None)  # Full load for inference
    dim_org_df = create_unified_schema(dim_org_raw, 'dim_org')
    dim_org_df, _ = filter_bad_records(dim_org_df)
    dim_org_df = create_blocking_keys(dim_org_df)
    print(f"Loaded dim_org: {len(dim_org_df):,} records")


---
## 2. Define Chunking Strategy


In [ ]:
# Get source tables for chunking
source_tables_query = """
    SELECT 
        source_table,
        COUNT(*) as record_count
    FROM allsci_prod_gold.potential_mismatched_organizations
    GROUP BY source_table
    ORDER BY record_count DESC
"""

source_tables_df = db.execute_query(source_tables_query, "Source table counts")
print("\nSOURCE TABLES FOR CHUNKING")
print("=" * 60)
print(source_tables_df.to_string())
print(f"\nTotal records: {source_tables_df['record_count'].sum():,}")


In [ ]:
# Compute IDF scores from reference data
# These are used to extract distinctive tokens during inference
idf_scores = compute_token_statistics(dim_org_df, 'name_normalized')
corpus_stopwords = get_corpus_stopwords(idf_scores, percentile=0.10)

print(f"\nIDF SCORES COMPUTED")
print("=" * 50)
print(f"Vocabulary size: {len(idf_scores):,} tokens")
print(f"Corpus stopwords: {len(corpus_stopwords):,}")

In [ ]:
# Configure chunking
# For demo, limit to first N records per source
# Set to None for full processing

DEMO_MODE = True  # Set to False for full processing
DEMO_LIMIT_PER_SOURCE = 1000 if DEMO_MODE else None

chunks = []
for _, row in source_tables_df.iterrows():
    chunks.append({
        'source_table': row['source_table'],
        'total_records': row['record_count'],
        'limit': DEMO_LIMIT_PER_SOURCE
    })

print(f"\nCHUNKING CONFIGURATION")
print("=" * 50)
print(f"Demo mode: {DEMO_MODE}")
print(f"Chunks: {len(chunks)}")
if DEMO_MODE:
    print(f"Limit per chunk: {DEMO_LIMIT_PER_SOURCE:,}")


---
## 3. Process Chunks


In [ ]:
def process_chunk(chunk_info, dim_org_df, model_json, db, idf_scores, corpus_stopwords):
    """Process a single chunk of mismatched records
    
    Returns:
        tuple: (predictions_df, source_id_mapping_df)
    """
    source_table = chunk_info['source_table']
    limit = chunk_info['limit']
    
    log_step(f"Processing: {source_table}")
    
    # Load chunk
    chunk_df = load_mismatched(db, source_table=source_table, limit=limit)
    
    if len(chunk_df) == 0:
        log_step(f"  No records to process", "WARN")
        return pd.DataFrame(), pd.DataFrame()
    
    # Prepare data
    chunk_unified = create_unified_schema(chunk_df, 'mismatched')
    
    # Collect source_entity_id mapping before Splink drops it
    source_id_mapping = chunk_unified[['unique_id', 'source_entity_id']].copy()
    
    chunk_filtered, removed = filter_bad_records(chunk_unified)
    
    if len(chunk_filtered) == 0:
        log_step(f"  All records filtered out", "WARN")
        return pd.DataFrame(), pd.DataFrame()
    
    chunk_prepared = create_blocking_keys(chunk_filtered)
    
    # Add distinctive tokens using pre-computed IDF scores
    chunk_prepared = add_distinctive_tokens(chunk_prepared, idf_scores, corpus_stopwords)
    
    # Add token set features (name_tokens, all_names)
    chunk_prepared = add_token_set_features(chunk_prepared)
    
    log_step(f"  Prepared: {len(chunk_prepared):,} records (filtered {len(removed):,})")
    
    # Remove training metadata (not a Splink setting)
    model_settings = {k: v for k, v in model_json.items() if k != 'training_metadata'}
    
    # Initialize linker with model
    linker = Linker(
        [dim_org_df, chunk_prepared],
        model_settings,
        db_api=DuckDBAPI()
    )
    
    # Predict
    predictions = linker.inference.predict(
        threshold_match_probability=config.matching.THRESHOLD_PREDICTION
    )
    predictions_df = predictions.as_pandas_dataframe()
    
    log_step(f"  Predictions: {len(predictions_df):,}")
    
    predictions_df['source_table'] = source_table
    return predictions_df, source_id_mapping

In [ ]:
# Load reference data (dim_org) for linking
cached_dim_org = load_checkpoint(config.paths.DATA_DIR + "/dim_org_training.parquet", "dim_org cache")

if cached_dim_org is not None:
    dim_org_df = cached_dim_org
    print(f"Loaded cached dim_org: {len(dim_org_df):,} records")
else:
    dim_org_raw = load_dim_org(db, sample_size=None)
    dim_org_df = create_unified_schema(dim_org_raw, 'dim_org')
    dim_org_df, _ = filter_bad_records(dim_org_df)
    dim_org_df = create_blocking_keys(dim_org_df)
    print(f"Loaded dim_org: {len(dim_org_df):,} records")

# Ensure dim_org has distinctive token features
if 'distinctive_token' not in dim_org_df.columns:
    print("Adding distinctive tokens to dim_org...")
    dim_org_df = add_distinctive_tokens(dim_org_df, idf_scores, corpus_stopwords)
    dim_org_df = add_token_set_features(dim_org_df)

In [ ]:
# Process all chunks
all_predictions = []
all_source_id_mappings = []  # Collect source_entity_id mappings
tracker = ProgressTracker(len(chunks), "Inference Pipeline")

for i, chunk_info in enumerate(chunks):
    tracker.step(f"{chunk_info['source_table']} ({chunk_info['total_records']:,} total)")
    
    try:
        chunk_predictions, source_id_mapping = process_chunk(chunk_info, dim_org_df, model_json, db, idf_scores, corpus_stopwords)
        if len(chunk_predictions) > 0:
            all_predictions.append(chunk_predictions)
            all_source_id_mappings.append(source_id_mapping)
        
        # Running statistics
        total_so_far = sum(len(p) for p in all_predictions)
        log_step(f"  Running total: {total_so_far:,} predictions")
        log_step(f"  ETA: {tracker.eta()}")
        
    except Exception as e:
        log_step(f"  Error: {str(e)}", "ERROR")
        continue

tracker.complete()

In [ ]:
sample_who = load_mismatched(db, source_table='who_clinical_trials_silver.studies_metadata', limit=100)
print(sample_who.columns.tolist())
multi = sample_who[sample_who['country'].str.contains(';', na=False)]
print(f"Records with multi-country: {len(multi)}")
if len(multi) > 0:
    print(multi['country'].value_counts().head(10))

In [ ]:
# Combine all predictions
if all_predictions:
    combined_predictions = pd.concat(all_predictions, ignore_index=True)
    
    # Join source_entity_id back to predictions
    if all_source_id_mappings:
        source_id_lookup = pd.concat(all_source_id_mappings, ignore_index=True).drop_duplicates()
        combined_predictions = combined_predictions.merge(
            source_id_lookup,
            left_on='unique_id_r',
            right_on='unique_id',
            how='left'
        )
        # Drop the duplicate unique_id column from the merge
        combined_predictions = combined_predictions.drop(columns=['unique_id'], errors='ignore')
        print(f"Joined source_entity_id: {combined_predictions['source_entity_id'].notna().sum():,} records")
    
    print(f"\nCOMBINED PREDICTIONS")
    print("=" * 50)
    print(f"Total predictions: {len(combined_predictions):,}")
    print(f"\nPredictions by source:")
    print(combined_predictions['source_table'].value_counts()) 
else:
    combined_predictions = pd.DataFrame()
    print("No predictions generated")


In [ ]:
# Disambiguate many-to-many matches
# Keep only the best match per mismatched record (unique_id_r)

print("DISAMBIGUATING MANY-TO-MANY MATCHES")
print("=" * 50)

# Count duplicates before disambiguation
dupes_per_mismatched = combined_predictions.groupby('unique_id_r').size()
multi_match_count = (dupes_per_mismatched > 1).sum()
print(f"Mismatched records with multiple matches: {multi_match_count:,}")

if multi_match_count > 0:
    # Tie-breaking criteria (in order of priority):
    # 1. Highest match_probability
    # 2. Country match (if country_code_r is available)
    # 3. Prefer shorter name (HQ entity over subsidiary with location suffix)
    # 4. Alphabetical (deterministic fallback)

    def add_tiebreaker_score(df):
        df = df.copy()
        
        # Country match bonus (if country available in mismatched)
        if 'country_code_r' in df.columns:
            df['country_bonus'] = (df['country_code_l'] == df['country_code_r']).astype(int)
        else:
            df['country_bonus'] = 0
        
        # Prefer shorter names (less specific = likely HQ)
        df['name_len'] = df['name_l'].str.len() if 'name_l' in df.columns else df['name_normalized_l'].str.len()
        
        return df

    combined_predictions = add_tiebreaker_score(combined_predictions)

    # Sort by tiebreakers, then keep first (best) per mismatched record
    combined_predictions_sorted = combined_predictions.sort_values(
        by=['unique_id_r', 'match_probability', 'country_bonus', 'name_len'],
        ascending=[True, False, False, True]
    )

    # Keep best match per mismatched record
    disambiguated = combined_predictions_sorted.groupby('unique_id_r').first().reset_index()

    print(f"\nBefore disambiguation: {len(combined_predictions):,} predictions")
    print(f"After disambiguation:  {len(disambiguated):,} predictions")
    print(f"Removed duplicates:    {len(combined_predictions) - len(disambiguated):,}")

    # Save full version before disambiguation (for debugging)
    save_checkpoint(combined_predictions, config.paths.DATA_DIR + "/predictions_all_matches.parquet", "All matches (before disambiguation)")
    
    # Use disambiguated version going forward
    combined_predictions = disambiguated

    # Clean up temp columns
    combined_predictions = combined_predictions.drop(columns=['country_bonus', 'name_len'], errors='ignore')
    
    print("\nDisambiguation complete - keeping best match per mismatched record")
else:
    print("No duplicate matches found - no disambiguation needed")


In [ ]:
combined_predictions

In [ ]:
query = """
SELECT *
FROM allsci_prod_gold.potential_mismatched_organizations
WHERE mismatch_id = '0024eb8b9d1aa30ded3f3bc9418fc7b8'
"""
record = db.execute_query(query, "Lookup specific record")
display(record)

In [ ]:
os.environ["ANTHROPIC_API_KEY"] = ""

---
## 3.5. Cluster Predictions

Build entity clusters using graph-based connected components before LLM validation.
The clustering is required for active learning sample selection.


In [ ]:
# Cluster predictions at threshold using graph-based approach
if len(combined_predictions) > 0:
    print("CLUSTERING PREDICTIONS")
    print("=" * 50)
    
    import networkx as nx
    
    # Filter to threshold
    high_conf = combined_predictions[
        combined_predictions['match_probability'] >= config.matching.CLUSTER_THRESHOLD
    ]
    
    print(f"Building graph from {len(high_conf):,} edges above threshold {config.matching.CLUSTER_THRESHOLD}")
    
    # Build graph from pairwise predictions
    G = nx.Graph()
    
    # Add edges for each prediction above threshold
    for _, row in high_conf.iterrows():
        G.add_edge(row['unique_id_l'], row['unique_id_r'], 
                   weight=row['match_probability'])
    
    # Find connected components (clusters)
    components = list(nx.connected_components(G))
    
    # Create clusters dataframe
    cluster_records = []
    for cluster_id, members in enumerate(components):
        for unique_id in members:
            cluster_records.append({
                'unique_id': unique_id,
                'cluster_id': cluster_id
            })
    
    clusters_df = pd.DataFrame(cluster_records)
    
    # Cluster statistics
    cluster_sizes = clusters_df.groupby('cluster_id').size()
    singleton_count = (cluster_sizes == 1).sum()
    multi_count = (cluster_sizes > 1).sum()
    
    print(f"\nClustering threshold: {config.matching.CLUSTER_THRESHOLD}")
    print(f"Total clusters: {len(cluster_sizes):,}")
    print(f"  Singleton clusters (1 record):  {singleton_count:,}")
    print(f"  Multi-record clusters (2+):     {multi_count:,}")
    print(f"  Largest cluster:                {cluster_sizes.max()} records")
    
    # Merge cluster_id back to predictions
    combined_predictions = combined_predictions.merge(
        clusters_df.rename(columns={'unique_id': 'unique_id_l'}),
        on='unique_id_l',
        how='left'
    )
    
    # Save clusters
    save_checkpoint(clusters_df, config.paths.CLUSTERS, "Entity clusters")
    print(f"\nClusters saved to: {config.paths.CLUSTERS}")
else:
    print("No predictions to cluster")


---
## 3.6. Hierarchy Roll-up

For predictions where the source record lacks geographic context (no country/city),
substitute region-specific subsidiaries with their parent organizations.

This prevents matching "Texas Instruments" to "Texas Instruments (Japan)" when we
don't know the source location.


In [ ]:
# Load organization hierarchy data
print("LOADING ORGANIZATION HIERARCHY")
print("=" * 50)

hierarchy_query = """
SELECT 
    from_organization_allsci_id as child_id,
    to_organization_allsci_id as parent_id
FROM allsci_prod_gold.fact_organization_hierarchy_organization
WHERE relationship_type = 'parent'
"""

hierarchy_df = db.execute_query(hierarchy_query, "Organization hierarchy")

print(f"Loaded {len(hierarchy_df):,} parent-child relationships")


In [ ]:
# Apply hierarchy roll-up
from data_prep import rollup_to_parent

if len(combined_predictions) > 0 and len(hierarchy_df) > 0:
    print("HIERARCHY ROLL-UP")
    print("=" * 50)
    
    combined_predictions, rollup_stats = rollup_to_parent(
        combined_predictions,
        hierarchy_df,
        dim_org_df
    )
    
    print(f"\nRoll-up complete:")
    print(f"  Predictions checked: {rollup_stats['total_checked']:,}")
    print(f"  Rolled up to parent: {rollup_stats['rolled_up']:,}")
    
    if rollup_stats['rolled_up'] > 0:
        print(f"\n  Sample substitutions:")
        for detail in rollup_stats.get('rollup_details', [])[:5]:
            print(f"    {detail['original_name'][:35]} -> {detail['parent_name'][:35]}")
else:
    print("Skipping hierarchy roll-up (no predictions or no hierarchy data)")


In [ ]:
# ============================================================================
# LLM VALIDATION - CRITICAL FIX
# ============================================================================
# This section was missing from the original pipeline!
# LLM judge was implemented but never called, resulting in 100% errors.

if config.llm_judge.ENABLE_LLM_VALIDATION:
    from anthropic import Anthropic
    from llm_judge import judge_match
    import os
    
    # Initialize Anthropic client
    api_key = os.environ.get("ANTHROPIC_API_KEY")
    if not api_key:
        log_step("WARNING: ANTHROPIC_API_KEY not set - skipping LLM validation", "WARN")
        log_step("Set with: export ANTHROPIC_API_KEY='your-key'", "WARN")
    else:
        client = Anthropic(api_key=api_key)
        
        print("\n" + "=" * 70)
        print("LLM VALIDATION PIPELINE")
        print("=" * 70)
        
        # ========================================================================
        # OPTION 1: MULTI-AGENT ROUTING (RECOMMENDED - 60% cost reduction)
        # ========================================================================
        if config.multi_agent.ENABLE_MULTI_AGENT:
            from multi_agent_validator import batch_validate_with_agents
            
            log_step("Using multi-agent routing for cost-efficient validation...")
            print(f"  Config: {config.llm_judge.PRIMARY_MODEL}")
            print(f"  Multi-agent routing: Enabled")
            print(f"  Predictions to validate: {len(combined_predictions):,}")
            
            # Apply active learning first if enabled
            if config.active_learning.ENABLE_ACTIVE_LEARNING:
                from active_learning import select_active_learning_batch
                
                log_step(f"Active learning: Selecting {config.active_learning.VALIDATION_BUDGET} highest-value samples...")
                
                validation_batch = select_active_learning_batch(
                    combined_predictions,
                    clusters_df,
                    budget=config.active_learning.VALIDATION_BUDGET,
                    min_per_source=config.active_learning.MIN_SAMPLES_PER_SOURCE,
                    source_weights=config.active_learning.SOURCE_WEIGHTS
                )
                
                print(f"  Selected {len(validation_batch):,} samples for validation")
                print(f"  Cost savings: {100 * (1 - len(validation_batch)/len(combined_predictions)):.1f}%")
            else:
                validation_batch = combined_predictions
                log_step("Validating all predictions (no active learning)")
            
            # Multi-agent validation
            combined_predictions, agent_stats = batch_validate_with_agents(
                validation_batch,
                dim_org_df,
                client,
                llm_judge_func=judge_match,
                enable_routing=True,
                primary_model=config.llm_judge.PRIMARY_MODEL,
                fast_model=config.llm_judge.FAST_MODEL
            )
            
            print("\nMULTI-AGENT STATISTICS:")
            print("=" * 70)
            print(f"Total validations: {agent_stats['total']:,}")
            if 'summary' in agent_stats:
                summary = agent_stats['summary']
                print(f"  Free (deterministic):  {summary['free_deterministic']:,} ({summary['cost_reduction_pct']:.1f}% savings)")
                print(f"  LLM calls required:    {summary['llm_required']:,}")
        
        # ========================================================================
        # OPTION 2: ACTIVE LEARNING ONLY (No multi-agent)
        # ========================================================================
        elif config.active_learning.ENABLE_ACTIVE_LEARNING:
            from active_learning import select_active_learning_batch
            from llm_judge import batch_judge_matches
            
            log_step(f"Using active learning: selecting {config.active_learning.VALIDATION_BUDGET} samples...")
            
            # Select high-value samples
            selected_batch = select_active_learning_batch(
                combined_predictions,
                clusters_df,
                budget=config.active_learning.VALIDATION_BUDGET,
                min_per_source=config.active_learning.MIN_SAMPLES_PER_SOURCE
            )
            
            print(f"  Selected: {len(selected_batch):,}/{len(combined_predictions):,} predictions")
            print(f"  Cost savings: {100 * (1 - len(selected_batch)/len(combined_predictions)):.1f}%")
            
            # Validate with LLM
            log_step(f"Validating {len(selected_batch):,} predictions with LLM...")
            
            llm_results = batch_judge_matches(
                selected_batch,
                dim_org_df,
                client,
                model=config.llm_judge.PRIMARY_MODEL,
                max_workers=config.llm_judge.MAX_WORKERS,
                enable_bias_mitigation=config.llm_judge.ENABLE_POSITION_BIAS_MITIGATION,
                enable_ensemble=config.llm_judge.ENABLE_ENSEMBLE,
                progress_callback=lambda i, total: print(f"  Progress: {i}/{total}", end='\r') if i % 50 == 0 else None
            )
            
            # Merge results back
            for result in llm_results:
                idx = result['prediction_idx']
                combined_predictions.loc[idx, 'llm_match'] = result['llm_match']
                combined_predictions.loc[idx, 'llm_confidence'] = result['llm_confidence']
                combined_predictions.loc[idx, 'llm_reason'] = result['llm_reason']
                if 'position_bias_detected' in result:
                    combined_predictions.loc[idx, 'position_bias_detected'] = result['position_bias_detected']
            
            log_step(f"LLM validation complete: {len(llm_results):,} results")
        
        # ========================================================================
        # OPTION 3: VALIDATE ALL (Most expensive)
        # ========================================================================
        else:
            from llm_judge import batch_judge_matches
            
            log_step(f"Validating ALL {len(combined_predictions):,} predictions with LLM...")
            log_step("WARNING: This is expensive! Consider enabling active_learning", "WARN")
            
            llm_results = batch_judge_matches(
                combined_predictions,
                dim_org_df,
                client,
                model=config.llm_judge.PRIMARY_MODEL,
                max_workers=config.llm_judge.MAX_WORKERS,
                enable_bias_mitigation=config.llm_judge.ENABLE_POSITION_BIAS_MITIGATION,
                enable_ensemble=config.llm_judge.ENABLE_ENSEMBLE,
                progress_callback=lambda i, total: print(f"  Progress: {i}/{total}", end='\r') if i % 100 == 0 else None
            )
            
            # Merge results
            for result in llm_results:
                idx = result['prediction_idx']
                combined_predictions.loc[idx, 'llm_match'] = result['llm_match']
                combined_predictions.loc[idx, 'llm_confidence'] = result['llm_confidence']
                combined_predictions.loc[idx, 'llm_reason'] = result['llm_reason']
            
            log_step(f"LLM validation complete: {len(llm_results):,} results")
        
        # ========================================================================
        # CONFIDENCE CALIBRATION (if enabled)
        # ========================================================================
        if config.llm_judge.ENABLE_CALIBRATION and 'llm_confidence' in combined_predictions.columns:
            from llm_judge import calibrate_confidence
            
            log_step("Calibrating LLM confidence scores...")
            
            try:
                calibrated = calibrate_confidence(combined_predictions)
                combined_predictions['llm_confidence_calibrated'] = calibrated
                log_step("Calibration complete")
            except Exception as e:
                log_step(f"Calibration failed: {str(e)}", "WARN")
        
        # ========================================================================
        # VALIDATION SUMMARY
        # ========================================================================
        print("\n" + "=" * 70)
        print("LLM VALIDATION SUMMARY")
        print("=" * 70)
        
        if 'llm_match' in combined_predictions.columns:
            validated = combined_predictions['llm_match'].notna()
            total_validated = validated.sum()
            confirmed = (combined_predictions['llm_match'] == True).sum()
            rejected = (combined_predictions['llm_match'] == False).sum()
            errors = (combined_predictions['llm_match'].isna()).sum()
            
            print(f"\nValidated: {total_validated:,}/{len(combined_predictions):,} predictions")
            print(f"  Confirmed matches: {confirmed:,} ({100*confirmed/total_validated:.1f}%)")
            print(f"  Rejected matches:  {rejected:,} ({100*rejected/total_validated:.1f}%)")
            if errors > 0:
                print(f"  Errors:            {errors:,}")
            
            # Disagreements
            if confirmed + rejected > 0:
                high_splink_rejected = combined_predictions[
                    (combined_predictions['match_probability'] >= 0.95) &
                    (combined_predictions['llm_match'] == False)
                ]
                
                low_splink_confirmed = combined_predictions[
                    (combined_predictions['match_probability'] < 0.70) &
                    (combined_predictions['llm_match'] == True)
                ]
                
                print(f"\nDisagreements:")
                print(f"  High Splink (>=0.95) but LLM rejected: {len(high_splink_rejected):,}")
                print(f"  Low Splink (<0.70) but LLM confirmed:  {len(low_splink_confirmed):,}")
                
                if len(high_splink_rejected) > 0 or len(low_splink_confirmed) > 0:
                    print(f"\n  → Consider reviewing disagreements for feedback!")
        
        print("\n" + "=" * 70)
        log_step("LLM validation pipeline complete!")

else:
    log_step("LLM validation disabled in config", "WARN")
    print("  To enable: config.llm_judge.ENABLE_LLM_VALIDATION = True")

---
## 3.5. LLM Validation with Multi-Agent Routing

CRITICAL FIX: This cell was missing - LLM validation was configured but never executed!

Now using multi-agent routing for cost-efficient validation:
- Active learning: Select highest-value samples (75% cost reduction)
- Multi-agent: Route to specialized agents (60% fewer LLM calls)
- Bias mitigation: Position bias detection (optional)
- Calibration: Confidence score calibration

In [ ]:
# [Cell moved to earlier position - clustering now runs before LLM validation]

---
## 4. Prediction Statistics


In [ ]:
# Prediction score distribution
if len(combined_predictions) > 0:
    print("PREDICTION SCORE DISTRIBUTION")
    print("=" * 50)
    
    bins = [0, 0.5, 0.7, 0.85, 0.95, 1.0]
    labels = ['0.5-0.7 (Low)', '0.7-0.85 (Medium)', '0.85-0.95 (High)', '0.95-1.0 (Very High)']
    combined_predictions['confidence_tier'] = pd.cut(
        combined_predictions['match_probability'], 
        bins=bins[1:], 
        labels=labels
    )
    
    print("\nConfidence tiers:")
    tier_counts = combined_predictions['confidence_tier'].value_counts().sort_index()
    for tier, count in tier_counts.items():
        pct = 100 * count / len(combined_predictions)
        print(f"  {tier:<30} | {count:>8,} ({pct:5.1f}%)")


In [ ]:
combined_predictions.to_csv('predictions.csv')

In [ ]:
import os



---
## 5. LLM Validation (Optional)

Use Claude to validate predictions with source-aware context.


In [ ]:
# LLM Validation Configuration
import os

ENABLE_LLM_VALIDATION = True  # Set to False to skip LLM validation
LLM_SAMPLE_SIZE = None  # None = all predictions, or int for sample
LLM_MODEL = "claude-sonnet-4-20250514"  # or "claude-3-5-haiku-20241022" for cheaper

# Set your Anthropic API key (uncomment one option):
# Option 1: Set directly (not recommended for shared notebooks)

# Option 2: Load from .env file or shell environment (recommended)

# Check if API key is set





In [ ]:
# LLM Validation Summary
if 'llm_match' in combined_predictions.columns:
    print("LLM VALIDATION SUMMARY")
    print("=" * 60)
    
    # Overall stats
    total = len(combined_predictions)
    llm_true = (combined_predictions['llm_match'] == True).sum()
    llm_false = (combined_predictions['llm_match'] == False).sum()
    llm_error = combined_predictions['llm_match'].isna().sum()
    
    print(f"\nOverall Results ({total:,} predictions):")
    print(f"  LLM Confirmed Match:    {llm_true:>6,} ({100*llm_true/total:5.1f}%)")
    print(f"  LLM Rejected:           {llm_false:>6,} ({100*llm_false/total:5.1f}%)")
    print(f"  LLM Errors:             {llm_error:>6,} ({100*llm_error/total:5.1f}%)")
    
    # By source table
    print("\nBy Source Table:")
    by_source = combined_predictions.groupby('source_table').agg({
        'llm_match': lambda x: (x == True).sum(),
        'match_probability': 'count'
    }).rename(columns={'llm_match': 'confirmed', 'match_probability': 'total'})
    by_source['rejected'] = by_source['total'] - by_source['confirmed']
    by_source['confirm_rate'] = 100 * by_source['confirmed'] / by_source['total']
    print(by_source.sort_values('total', ascending=False).to_string())
    
    # Disagreements (high Splink score but LLM rejected)
    disagreements = combined_predictions[
        (combined_predictions['match_probability'] >= 0.95) & 
        (combined_predictions['llm_match'] == False)
    ]
    print(f"\nDisagreements (Splink >= 0.95 but LLM rejected): {len(disagreements):,}")
    
    if len(disagreements) > 0:
        print("\nSample disagreements:")
        sample_cols = ['name_l', 'name_r', 'match_probability', 'llm_confidence', 'llm_reason', 'source_table']
        available_cols = [c for c in sample_cols if c in disagreements.columns]
        print(disagreements[available_cols].head(10).to_string())


---
## 6. Export Results


In [ ]:
# Save predictions
if len(combined_predictions) > 0:
    save_checkpoint(combined_predictions, config.paths.PREDICTIONS, "All predictions")
    
    # Save LLM-validated predictions separately if available
    if 'llm_match' in combined_predictions.columns:
        # Confirmed matches (LLM agrees)
        confirmed = combined_predictions[combined_predictions['llm_match'] == True]
        if len(confirmed) > 0:
            save_checkpoint(
                confirmed, 
                config.paths.DATA_DIR + "/llm_confirmed_matches.parquet", 
                "LLM confirmed matches"
            )
        
        # Rejected matches (LLM disagrees)
        rejected = combined_predictions[combined_predictions['llm_match'] == False]
        if len(rejected) > 0:
            save_checkpoint(
                rejected, 
                config.paths.DATA_DIR + "/llm_rejected_matches.parquet", 
                "LLM rejected matches"
            )
    
    # Save high-confidence matches separately
    high_conf = combined_predictions[combined_predictions['match_probability'] >= config.matching.THRESHOLD_HIGH_CONFIDENCE]
    if len(high_conf) > 0:
        save_checkpoint(high_conf, config.paths.DATA_DIR + "/high_confidence_matches.parquet", "High confidence matches")
    
    print(f"\nResults saved:")
    print(f"  - All predictions: {config.paths.PREDICTIONS}")
    print(f"  - High confidence: {config.paths.DATA_DIR}/high_confidence_matches.parquet")


In [ ]:
# Summary
print("\n" + "=" * 70)
print("INFERENCE COMPLETE")
print("=" * 70)

if len(combined_predictions) > 0:
    llm_info = ""
    if 'llm_match' in combined_predictions.columns:
        confirmed = (combined_predictions['llm_match'] == True).sum()
        rejected = (combined_predictions['llm_match'] == False).sum()
        llm_info = f"""
LLM VALIDATION:
  - LLM Confirmed: {confirmed:,}
  - LLM Rejected: {rejected:,}
  - Confirm Rate: {100*confirmed/(confirmed+rejected):.1f}%"""
    
    print(f"""
RESULTS SUMMARY:
  - Total predictions: {len(combined_predictions):,}
  - High confidence (>0.95): {(combined_predictions['match_probability'] > 0.95).sum():,}
  - Sources processed: {len(chunks)}{llm_info}

OUTPUT FILES:
  - All predictions: {config.paths.PREDICTIONS}
  - High confidence: {config.paths.DATA_DIR}/high_confidence_matches.parquet""")
    
    if 'llm_match' in combined_predictions.columns:
        print(f"""  - LLM confirmed: {config.paths.DATA_DIR}/llm_confirmed_matches.parquet
  - LLM rejected: {config.paths.DATA_DIR}/llm_rejected_matches.parquet""")
    
    print("""
NEXT STEPS:
  1. Run 4_analysis.ipynb to review results
  2. Review LLM rejected matches for false positives
  3. Export final matches for production use
""")

# Cleanup
db.close()
print("Database connection closed")


In [ ]:
df.to_csv('test.csv')